<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>AI SYSTEMS ENGINEERING 1</b>

## Unit 4: Complete RAG Pipeline

**CV Raman Global University, Bhubaneswar**  
*AI Center of Excellence*

</div>

---

You've learned embeddings, chunking, and vector search in the previous notebook. Now let's combine them with an LLM to build a complete RAG chatbot.

### What You'll Learn

1. **Build a generator** that uses retrieved context to answer questions
2. **Assemble a complete RAG pipeline** (retriever + generator)
3. **Create a chat interface** with Gradio

---

## 1. Setup

In [ ]:
# Install required packages
!pip install -q openai chromadb sentence-transformers gradio langchain-text-splitters

import os
from getpass import getpass
from openai import OpenAI
from sentence_transformers import SentenceTransformer
import chromadb
import gradio as gr

# Configure API
api_key = getpass("Enter your OpenAI API Key: ")
os.environ['OPENAI_API_KEY'] = api_key
client = OpenAI(api_key=api_key)
MODEL = "gpt-4o-mini"

# Load embedding model and initialize vector store
embedder = SentenceTransformer('all-MiniLM-L6-v2')
chroma = chromadb.Client()

print("Setup complete")

---

## 2. RAG Pipeline Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                     RAG Pipeline                            │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  [Documents] → [Chunking] → [Embeddings] → [Vector DB]     │
│                                                ↓            │
│  [User Query] → [Query Embedding] → [Similarity Search]     │
│                                                ↓            │
│  [Retrieved Context] + [Query] → [LLM] → [Answer]          │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

---

## 3. Knowledge Base

Let's chunk a company document and index it for retrieval.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

company_info = """
TechSolutions India was founded in 2018 by Priya Sharma and Rahul Verma. The company is headquartered in Bhubaneswar, Odisha, with additional offices in Bangalore and Hyderabad. TechSolutions specializes in AI/ML solutions, cloud services, and mobile applications, and has grown to over 250 employees. The company achieved ₹50 crores in annual revenue in 2024.

Priya Sharma is the CEO and co-founder. She graduated from IIT Delhi and completed her MBA from Stanford University. She previously worked as Senior Director at Infosys and won the Women in Tech Leader award in 2022. Rahul Verma is the CTO and co-founder. He graduated from BITS Pilani and previously worked as Tech Lead at Google India. His expertise is in Machine Learning and Cloud Architecture. Ananya Patel is the VP of Engineering and manages a team of 100+ engineers.

TechSolutions offers three main products. CloudAssist Pro is the flagship enterprise cloud management platform, costing ₹50,000 per month, and includes auto-scaling, real-time monitoring, cost optimization, and 24/7 support. SmartHR is an AI-powered HR management system at ₹25,000 per month that handles recruitment automation, payroll processing, performance tracking, and employee analytics. DataViz Analytics is a business intelligence platform at ₹15,000 per month with real-time dashboards, custom reports, and predictive analytics.

Work hours at TechSolutions are 9 AM to 6 PM, Monday to Friday, following a hybrid model with 3 days in office and 2 days remote. Employees receive 24 paid leaves and 10 sick leaves per year. Maternity leave is 26 weeks and paternity leave is 2 weeks. Probation period is 6 months for all new employees, with a notice period of 2 months for permanent employees and 1 month during probation.

Employee benefits include health insurance coverage of ₹5 lakh for employees and their families, covering hospitalization, OPD, dental, and vision. The learning budget is ₹50,000 per year per employee for courses, certifications, and conferences. Performance bonuses can be up to 20% of annual salary. Major clients include HDFC Bank, Tata Motors, Reliance Industries, and ICICI Bank, with a 95% customer satisfaction rate across 200+ completed projects.
"""

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
chunks = splitter.split_text(company_info)

collection = chroma.create_collection("knowledge_base")
embeddings = embedder.encode(chunks)

collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    embeddings=embeddings.tolist(),
    documents=chunks
)

print(f"Indexed {collection.count()} chunks")

---

## 4. Retriever

In [ ]:
def retrieve(query, n_results=3):
    """Retrieve relevant documents for a query."""
    query_embedding = embedder.encode([query])
    
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=n_results
    )
    
    return results['documents'][0]

# Quick test
query = "Who is the CEO?"
docs = retrieve(query)

print(f"Query: {query}\n")
print("Retrieved documents:")
for i, doc in enumerate(docs):
    print(f"{i+1}. {doc[:100]}...")

---

## 5. Generator

In [ ]:
SYSTEM_PROMPT = """
You are a helpful assistant for TechSolutions India.
Answer questions based on the provided context.
Be accurate and concise. If the answer is not in the context, say so.

Context:
{context}
"""

def generate_answer(query, context, history=[]):
    """Generate answer using LLM with retrieved context."""
    system_message = SYSTEM_PROMPT.format(context="\n\n".join(context))
    
    messages = [
        {"role": "system", "content": system_message}
    ] + history + [
        {"role": "user", "content": query}
    ]
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0
    )
    
    return response.choices[0].message.content

---

## 6. Complete RAG Function

In [ ]:
def rag_answer(query, history=[]):
    """Complete RAG pipeline: retrieve + generate."""
    # Step 1: Retrieve relevant documents
    context = retrieve(query, n_results=3)
    
    # Step 2: Generate answer
    answer = generate_answer(query, context, history)
    
    return answer

In [ ]:
# Test the RAG pipeline
test_queries = [
    "Who founded TechSolutions and when?",
    "How much does CloudAssist Pro cost?",
    "What is the leave policy?",
    "What benefits do employees get?",
    "Who are the major clients?"
]

for query in test_queries:
    print(f"Q: {query}")
    answer = rag_answer(query)
    print(f"A: {answer}\n")

---

## 7. Chat Interface

In [ ]:
def chat(message, history):
    """Chat function for Gradio."""
    return rag_answer(message, history)

demo = gr.ChatInterface(
    chat,
    title="TechSolutions AI Assistant",
    description="Ask questions about TechSolutions India - company, products, policies, and more.",
    examples=[
        "Who is the CEO of TechSolutions?",
        "What products does the company offer?",
        "What is the work from home policy?",
        "How much is the learning budget?"
    ],
    type="messages"
)

demo.launch()

---

## Key Takeaways

1. **RAG = Retrieval + Generation** - simple but powerful pattern

2. **Embeddings enable semantic search** - find relevant docs by meaning

3. **Context improves accuracy** - LLM answers grounded in documents

4. **Easy to customize** - adjust prompts, retrieval, and models

### Pipeline Components

| Component | Role |
|-----------|------|
| Embedder | Converts text to vectors |
| Vector DB | Stores and searches vectors |
| Retriever | Finds relevant documents |
| Generator | Creates answers from context |

### What's Next?

In the next notebook, we'll learn how to evaluate RAG performance.

---

## Additional Resources

- [Chroma Documentation](https://docs.trychroma.com/)
- [RAG Best Practices](https://www.pinecone.io/learn/retrieval-augmented-generation/)

---

**Course Information:**
- **Institution:** CV Raman Global University, Bhubaneswar
- **Program:** AI Center of Excellence
- **Course:** AI Systems Engineering 1
- **Developed by:** [Poorit Technologies](https://poorit.in) - *Transform Graduates into Industry-Ready Professionals*

---